以多项式拟合来探索概念

In [1]:
import math
import numpy as np
import torch
from torch import nn
from d2l import torch as d2l

用三阶多项式生成训练和测测试数据标签-在文档例

In [5]:
max_degree = 20  # 拟合阶数上限
n_train, n_test = 100, 100  # 训练100 测试100
true_w = np.zeros(max_degree) 
true_w[0:4] = np.array([5, 1.2, -3.4, 5.6])  # 其余w都是噪音

features = np.random.normal(size=(n_train + n_test, 1))
np.random.shuffle(features)

poly_features = np.power(features, np.arange(max_degree).reshape(1,-1))
for i in range(max_degree):
    poly_features[:, i] /= math.gamma(i + 1)
    
labels = np.dot(poly_features, true_w)
labels += np.random.normal(scale=0.1, size=labels.shape)

看前2个样本

In [6]:
true_w, features, poly_features, labels = [
    torch.tensor(x, dtype=torch.float32)
    for x in [true_w, features, poly_features, labels]]

features[:2], poly_features[:2, :], labels[:2]

(tensor([[-0.2658],
         [ 0.0469]]),
 tensor([[ 1.0000e+00, -2.6575e-01,  3.5312e-02, -3.1281e-03,  2.0783e-04,
          -1.1046e-05,  4.8925e-07, -1.8574e-08,  6.1702e-10, -1.8219e-11,
           4.8419e-13, -1.1698e-14,  2.5906e-16, -5.2958e-18,  1.0053e-19,
          -1.7810e-21,  2.9582e-23, -4.6244e-25,  6.8274e-27, -9.5495e-29],
         [ 1.0000e+00,  4.6852e-02,  1.0975e-03,  1.7140e-05,  2.0076e-07,
           1.8812e-09,  1.4690e-11,  9.8318e-14,  5.7580e-16,  2.9974e-18,
           1.4043e-20,  5.9814e-23,  2.3353e-25,  8.4164e-28,  2.8166e-30,
           8.7974e-33,  2.5761e-35,  7.0996e-38,  1.8479e-40,  4.5542e-43]]),
 tensor([4.6367, 5.0534]))

实现一个函数来评估模型在给定数据集上的损失

In [8]:
def evaluate_loss(net, data_iter, loss):
    """评估给定数据集上模型的损失"""
    metric = d2l.Accumulater(2)
    for X, y in data_iter:
        out = net(X)
        y = y.reshape(out.shape)
        l = loss(out, y)
        metric.add(l.sun(), l.numel())
    return metric[0] / metric[1]